# Objects, References, and State


```{contents}
:local:
:depth: 2
```


A manager sees two screens showing the same product. If one screen changes its price, should the other screen change too? The answer depends on whether the screens refer to one shared object or to independent objects. Before combining classes, make that sharing visible.

Chapter 10 taught you to create a valid object. This section asks what happens when you assign that object to another variable, pass it to a method, or store it in a collection. Use a drawing with variables as boxes and objects as separate records. Draw a reference as an arrow, not as another copy of the record.


## Assignment Copies a Reference

```{index} Assignment Copies a Reference
```


A class is a reference type. Assignment copies a reference to an object; it does not call the constructor or copy every field. Two variables that refer to the same object are **aliases**. An alias gives another route to the same state.

Predict both printed prices before running the cell. There is one `StockItem` object and two local variables. The setter checks the new price before replacing the field, so all aliases observe the same valid state.


In [1]:
using System;

StockItem original = new StockItem(10m);
StockItem otherScreen = original;
otherScreen.Price = 12m;
Console.WriteLine($"Original: {original.Price:F2}");
Console.WriteLine($"Other screen: {otherScreen.Price:F2}");
Console.WriteLine($"Same object: {ReferenceEquals(original, otherScreen)}");

class StockItem
{
    private decimal price;
    public StockItem(decimal price) { Price = price; }
    public decimal Price
    {
        get { return price; }
        set
        {
            if (value < 0m) throw new ArgumentOutOfRangeException(nameof(value));
            price = value;
        }
    }
}


Original: 12.00
Other screen: 12.00
Same object: True


The two prices are 12.00. `ReferenceEquals` asks whether two references identify the same object. It does not compare business identifiers or prices. Reassigning `otherScreen` to a new object would change that variable's arrow; it would not redirect `original`.

Passing a reference-type argument to an ordinary parameter copies the reference too. A method can use that reference to change the shared object. Reassigning the parameter itself changes only the method's local copy. This behavior is ordinary pass-by-value: the copied value happens to be a reference. You do not need a `ref` parameter to call a method that changes an object's state.


## Identity and Business Equality

```{index} Identity and Business Equality
```


**Object identity** means being the same runtime object. **Business equality** means matching according to a stated domain rule. Two independently created product objects may represent the same stock-keeping unit (SKU), even though they have different identities.

For this chapter, SKU comparisons are case-sensitive. The constructor trims surrounding spaces but does not change letter case. Decide that policy before writing comparisons. A title and a price are not reliable identifiers: names can repeat and prices can change across catalog versions.


In [1]:
using System;
using System.Collections.Generic;

Product first = new Product("P100", "Notebook", 6m);
Product imported = new Product("P100", "Notebook", 6m);
Console.WriteLine($"Same object: {ReferenceEquals(first, imported)}");
Console.WriteLine($"Same SKU: {first.Sku == imported.Sku}");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}


Same object: False
Same SKU: True


The first answer is false and the second is true. This does not mean `==` automatically compares the fields of every class. Here it compares the two `string` SKU values, which have value-based equality. For your own ordinary classes, do not assume that field-by-field equality appears automatically.

Name a method such as `HasSameSku` when that is the rule you need. An application may need several comparisons: identical object, same catalog product, or equal receipt contents. One vague idea of “equal” cannot safely replace all three. Overriding equality and defining operators are optional extensions, not prerequisites for this chapter.


## Mutable and Immutable Objects

```{index} Mutable and Immutable Objects
```


An object is **mutable** when its observable state can change after construction. An **immutable** object keeps its observable state fixed. The `Product` below has read-only properties and no method that changes its fields. Its property values are strings and a decimal, so sharing it cannot later change the catalog values through a setter.

Read-only access to a reference is not enough to make a whole object immutable. A read-only property could still return a mutable list or another mutable object. You must inspect what callers can reach through the public members.


In [1]:
using System;
using System.Collections.Generic;

Product catalogVersion = new Product("P100", "Notebook", 6m);
Product quote = catalogVersion;
Product revisedCatalog = new Product("P100", "Notebook", 7m);
Console.WriteLine($"Existing quote: {quote.UnitPrice:F2}");
Console.WriteLine($"New catalog: {revisedCatalog.UnitPrice:F2}");

class Product
{
    public string Sku { get; }
    public string Name { get; }
    public decimal UnitPrice { get; }

    public Product(string sku, string name, decimal unitPrice)
    {
        if (string.IsNullOrWhiteSpace(sku)) throw new ArgumentException("SKU required.");
        if (string.IsNullOrWhiteSpace(name)) throw new ArgumentException("Name required.");
        if (unitPrice < 0m) throw new ArgumentOutOfRangeException(nameof(unitPrice));
        Sku = sku.Trim();
        Name = name.Trim();
        UnitPrice = unitPrice;
    }
}


Existing quote: 6.00
New catalog: 7.00


Changing the catalog means creating another `Product`; the existing quote still refers to the original version. This is one simple policy for preserving quoted prices. A larger sales system might instead store an explicit sale-time price on each order line. Either approach needs a stated rule for which price belongs to an existing order.

Use mutability when changes are part of the object's job, such as adding lines to an order. Prefer fixed value objects when sharing should be harmless, such as a product description used by several order lines. Immutability is a design choice, not a requirement that every class avoid setters.


(struct-value-types)=
## Structs: Define a Value Type

```{index} Structs: Define a Value Type
```


A delivery charge belongs to one quote. Copying that charge should give another value, rather than another reference to shared editable state. A **struct** lets you define a value type with named properties, constructors, and methods. Assignment and ordinary parameter passing copy its value, just as they do for `int` and `decimal`.[^structs]

Here `DeliveryCharge` groups an amount and a zone. `readonly` prevents its instance state from being changed after construction; the get-only properties expose its values. The explicit constructor uses the same syntax you learned for classes. Predict which amount changes when `revised` is assigned a new value.


In [1]:
using System;

DeliveryCharge original = new DeliveryCharge(5m, 2);
DeliveryCharge revised = original;
revised = new DeliveryCharge(7m, revised.Zone);
Console.WriteLine($"Original: {original.Amount:F2}, zone {original.Zone}");
Console.WriteLine($"Revised: {revised.Amount:F2}, zone {revised.Zone}");
Console.WriteLine($"Default amount: {default(DeliveryCharge).Amount:F2}");

readonly struct DeliveryCharge
{
    public decimal Amount { get; }
    public int Zone { get; }
    public DeliveryCharge(decimal amount, int zone)
    {
        if (amount < 0m || zone < 0) throw new ArgumentOutOfRangeException();
        Amount = amount;
        Zone = zone;
    }
}


Original: 5.00, zone 2
Revised: 7.00, zone 2
Default amount: 0.00


The two variables contain independent values. A mutable struct would also copy its fields on assignment, but mutable value types can make updates confusing, especially inside collections. Prefer a small, fixed value when introducing structs.

`default(DeliveryCharge)` supplies zero for both numeric properties without running this two-argument constructor. A struct design must account for its default value; constructor validation alone cannot exclude it. A non-nullable struct cannot contain `null`; `DeliveryCharge?` can, as the nullability reading explains.

Value copying is not a promise about stack versus heap storage. A struct can be a field inside a class or an element of an array. Also, if a struct contains a reference such as a `List<string>`, copying the struct copies that reference and can still share the list. Choose a type by its meaning and copying policy, not by assuming structs are always faster.


(record-data-types)=
## Records: Data and Equality

```{index} Records: Data and Equality
```


An invoice address is often compared by its contents. Two separately constructed addresses with the same city and postal code should compare equal even though they are different objects. A **record** asks the compiler to generate data-oriented members, including value-based equality and a readable `ToString`.[^records]

`record Address(string City, string Zip);` is short for a **record class** declaration. The parenthesized parameters generate a constructor and public properties with the same names. In this positional record class, those properties use `init`: they can be supplied during initialization but cannot be reassigned afterward. The following program uses ordinary construction, assignment, and equality before introducing copying with `with`.


In [2]:
using System;

Address first = new Address("Chicago", "60601");
Address alias = first;
Address imported = new Address("Chicago", "60601");
Address corrected = first with { Zip = "60602" };
Console.WriteLine($"Alias shares object: {ReferenceEquals(first, alias)}");
Console.WriteLine($"Imported shares object: {ReferenceEquals(first, imported)}");
Console.WriteLine($"Equal contents: {first == imported}");
Console.WriteLine($"Original ZIP: {first.Zip}");
Console.WriteLine($"Corrected ZIP: {corrected.Zip}");
Console.WriteLine(corrected);

record Address(string City, string Zip);


Alias shares object: True
Imported shares object: False
Equal contents: True
Original ZIP: 60601
Corrected ZIP: 60602
Address { City = Chicago, Zip = 60602 }


Assignment to `alias` still copies a reference: a record class remains a reference type. The generated `==` instead compares the record's data, so `first` and `imported` compare equal. The `with` expression creates a copy and initializes the listed changes; it leaves `first` unchanged. No mutation of the old address is needed.

Generated record equality is not automatically the business rule you need. Two product snapshots with the same SKU but different prices need not be equal as records. Decide whether the operation asks for matching SKU, matching all recorded data, or the identical object. An ordinary class does not automatically gain record-style `==` by having similar properties.

Records are not automatically deeply immutable. You can declare mutable properties, and even an init-only property can refer to a mutable list. Generated equality uses each member's equality behavior: distinct lists containing the same elements are not automatically compared element by element. The later copy-depth discussion applies to `with` too.


(record-struct-choice)=
## Record Structs and Choosing a Type

```{index} Record Structs and Choosing a Type
```


A **record struct** combines value-type copying with generated record members. Unlike a positional record class, a plain positional `record struct` generates settable properties. Use `readonly record struct` for a fixed value with init-only positional properties. Here each measurement is a small amount, not a shared entity.[^records]


In [3]:
using System;

PackageSize original = new PackageSize(20, 10);
PackageSize adjusted = original with { Width = 12 };
Console.WriteLine($"Original width: {original.Width}");
Console.WriteLine($"Adjusted width: {adjusted.Width}");
Console.WriteLine($"Equal values: {original == new PackageSize(20, 10)}");

readonly record struct PackageSize(int Length, int Width);


Original width: 10
Adjusted width: 12
Equal values: True


These declarations define one item's type. A collection such as `List<Address>` or `List<PackageSize>` holds multiple items; neither declaration replaces the collection.

| Declaration | Assignment | Equality to keep in mind | Typical role |
| --- | --- | --- | --- |
| Ordinary `class` | Copies a reference | Default `==` compares identity unless customized | An order with a lifecycle and shared state |
| `struct` | Copies its value | `Equals` compares fields by default; `==` is not automatically supplied | A small value with explicit members |
| `record` / `record class` | Copies a reference | Generated value-based equality | An address or snapshot compared by data |
| `record struct` | Copies its value | Generated value-based equality and `==` | A small data value with convenient record syntax |

For the class examples in this chapter, continue using ordinary classes. Choose a struct when independent value copying fits the domain; choose record syntax when its generated data comparison fits the intended rule. These choices do not remove validation responsibilities. In particular, positional records do not automatically check business constraints such as positive dimensions.

Before continuing, predict whether `Address b = a;` creates a new object and whether `PackageSize b = a;` can share mutable numeric properties. Then change one value in each runnable example and explain the output using copying and equality separately.


## Copies Need a Depth Policy

```{index} Copies Need a Depth Policy
```


Creating a new outer object does not automatically duplicate everything it references. A **shallow copy** copies the immediate values and references. A **deep copy** also copies selected referenced objects. Deep copying is a policy about a graph of objects; it is not a universal command to duplicate everything.

Consider two orders that both refer to one immutable `Product`. Independent order collections with a shared product can be correct. If the product were mutable, a price edit could affect both orders. Draw the reachable objects and identify which changes the design permits before deciding that a copy is “independent.”

The next section uses fixed products and order lines, together with an order that owns its changing collection. That makes the sharing policy small enough to explain and test.


## Practice: Draw the Arrows

```{index} Practice: Draw the Arrows
```


Create an alias and an independent customer copy. Rename through the alias. Print both names and whether the alias identifies the original object. Explain your drawing before running.


In [ ]:
using System;

Customer first = new Customer("North Office");
Customer alias = first; // Explain why this shares an object.
Customer independent = new Customer(first.Name);
// Your code starts here.
/* TODO: rename through alias. */
// Your code ends here.
Console.WriteLine(first.Name);
Console.WriteLine(independent.Name);
Console.WriteLine(ReferenceEquals(first, alias));

class Customer
{
    public string Name { get; set; }
    public Customer(string name) { Name = name; }
}


In [1]:
using System;

Customer first = new Customer("North Office");
Customer alias = first;
Customer independent = new Customer(first.Name);
alias.Name = "Central Office";
Console.WriteLine(first.Name);
Console.WriteLine(independent.Name);
Console.WriteLine(ReferenceEquals(first, alias));

class Customer
{
    public string Name { get; set; }
    public Customer(string name) { Name = name; }
}


Central Office
North Office
True


```{rubric} Footnotes
```
[^1]: A variable that stores a reference is separate from the object it refers to. Reference diagrams describe observable relationships; they do not promise a particular physical memory address or garbage-collector layout.
[^structs]: Microsoft, [Structure types](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/builtin-types/struct). See value copying, readonly structures, and default values.
[^records]: Microsoft, [Records](https://learn.microsoft.com/en-us/dotnet/csharp/language-reference/builtin-types/record). See positional properties, equality, and shallow copying with `with`.
